In [20]:
# imports
import pyodbc
import pandas as pd
from pandas import DataFrame
from pyodbc import Row

In [ ]:
# configs
name_map: dict[str, str] = {
    "dow": "day_of_week",
    "dom": "day_of_month",
    "doy": "day_of_year",
    "wn": "week_number",
    "mn": "month_number",
    "qn": "quarter_number",
    "yn": "year_number",
}


def sql_grain_pt(grain: str) -> str:
    return f"""
        WITH {grain}_purchase_times AS (
            SELECT
                o.{grain},
                CONVERT(TIME, o.order_purchase_timestamp) AS purchase_time,
                DATEDIFF(SECOND, o.order_purchase_timestamp, order_approved_at)
                    AS diff_purchase_to_approve_s
            FROM sales.vw_orders_practical AS o
        )
        SELECT
            o.*,
            COUNT(*) AS order_count
        FROM {grain}_purchase_times AS o
        GROUP BY
            o.{grain},
            o.purchase_time,
            o.diff_purchase_to_approve_s
        ORDER BY
            o.{grain},
            o.purchase_time,
            o.diff_purchase_to_approve_s;
        """


# connect
driver = "ODBC Driver 17 for SQL Server"
server = "localhost"
database = "olist"

connstring: str = f"""
    DRIVER={{{driver}}};
    SERVER={server};
    DATABASE={database};
    Trusted_Connection=yes;
"""

conn = pyodbc.connect(connstring)
cursor = conn.cursor()

# get day table
day_pt_rows: list[Row] = cursor.execute("""
    SELECT
        CONVERT(TIME, o.order_purchase_timestamp) AS purchase_time,
        COUNT(*) AS order_count
    FROM sales.vw_orders_practical AS o
    GROUP BY CONVERT(TIME, o.order_purchase_timestamp)
    ORDER BY purchase_time;
""").fetchall()

day_pt_columns: list[str] = [description[0] for description in cursor.description]
df_pt_day: DataFrame = pd.DataFrame(
    (tuple(r) for r in day_pt_rows), columns=day_pt_columns
)

# get other tables
dfs_pt: dict[str, DataFrame] = {}

for name in name_map:
    df_name = f"{name}"
    grain = name_map[name]

    sql = sql_grain_pt(grain=grain)
    grain_pt_rows: list[Row] = cursor.execute(sql).fetchall()

    grain_pt_columns: list[str] = [description[0] for description in cursor.description]
    df_grain_pt = pd.DataFrame(
        (tuple(r) for r in grain_pt_rows), columns=grain_pt_columns
    )

    dfs_pt[df_name] = df_grain_pt

conn.close()


In [31]:
print(df_pt_day)
print(dfs_pt['dow'])


      purchase_time  order_count
0          00:00:00            1
1          00:00:01            2
2          00:00:02            1
3          00:00:06            3
4          00:00:07            1
...             ...          ...
50542      23:59:53            1
50543      23:59:54            4
50544      23:59:55            1
50545      23:59:58            3
50546      23:59:59            1

[50547 rows x 2 columns]
       day_of_week purchase_time  diff_purchase_to_approve_s  order_count
0                1      00:00:01                       914.0            1
1                1      00:00:01                      1526.0            1
2                1      00:00:13                     88497.0            2
3                1      00:00:29                      1120.0            1
4                1      00:00:35                    165496.0            1
...            ...           ...                         ...          ...
98274            7      23:59:42                         0.0